# Market Regime Allocation with Hidden Markov Models

## Table of Contents

1. [Project Motivation](#1.-Project-Motivation)
2. [Asset Universe and Data Sources](#2.-Asset-Universe-and-Data-Sources)
3. [Feature Engineering](#3.-Feature-Engineering)
4. [Hidden Markov Model Explanation](#4.-Hidden-Markov-Model-Explanation)
5. [Regime Model Training](#5.-Regime-Model-Training)

## 1. Project Motivation

Markets do not behave the same way all the time.

Sometimes the market is calm: equities trend upward, volatility is low, and investors are willing to take risk. At other times, the market becomes stressed: prices fall quickly, volatility rises, and assets that usually diversify a portfolio may stop behaving as expected.

This project is about building a systematic way to identify those changing market conditions and test whether they can support better portfolio allocation decisions.

### Simple intuition

A simple way to think about market regimes is to compare markets to weather.

If the weather is sunny, you dress differently than if there is a storm. You do not need to know the exact weather tomorrow to understand that an umbrella is useful when storm conditions are becoming more likely.

Market regimes are similar. The goal is not to predict every daily market move. The goal is to estimate the current environment, such as calm growth, high-volatility stress, or inflation/rate pressure, and then ask whether the portfolio should be positioned differently under those conditions.

### Why regimes matter

Many investment strategies implicitly assume that market relationships are reasonably stable. For example, a balanced portfolio often assumes that equities provide growth while bonds help reduce risk during equity selloffs.

In reality, these relationships can change. During some stress periods, volatility rises, risky assets fall together, and diversification becomes less reliable. During inflation or rate-hiking environments, long-duration bonds may not hedge equities as well as they do during growth shocks.

A good example is 2022. Many investors had become used to the idea that long-term Treasury bonds could help hedge equity drawdowns. This worked well in periods such as the dot-com crash and the 2008 Global Financial Crisis, where recession risk led central banks to cut interest rates. Falling interest rates usually push bond prices up, which can help offset equity losses.

In 2022, the situation was different. Inflation was high, and the Federal Reserve raised interest rates aggressively to bring inflation under control. Equities fell, but long-duration Treasury bonds also fell because yields were rising. The usual stock-bond hedge failed because the market environment was different: the dominant risk was not just recession risk, but inflation and rate pressure.

This matters because a portfolio that is appropriate in one environment may become too risky in another. A regime-aware process tries to recognize when the environment has changed instead of treating all historical periods as if they came from the same market condition.

### Research background

The idea of regime shifts is well established in economics and finance.

- Hamilton's classic Markov-switching model introduced a way to model economic time series as moving between hidden states, such as expansion and recession-like conditions. The important idea for this project is that these states are not directly observed. They are inferred from data as probabilities, not treated as objective labels. See Hamilton (1989), ["A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle"](https://www.econometricsociety.org/publications/econometrica/1989/03/01/new-approach-economic-analysis-nonstationary-time-series-and).
- Ang and Bekaert studied time-varying correlations and found evidence that volatility and correlations can behave differently across regimes. This supports the idea that risk itself can be state-dependent, rather than constant through time. See their NBER working paper, ["International Asset Allocation with Time-Varying Correlations"](https://www.nber.org/papers/w7056).
- Ang and Bekaert's ["How Do Regimes Affect Asset Allocation?"](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=310626) is especially relevant to this project because it connects regime shifts directly to portfolio allocation. The key motivation is not only to identify regimes, but to ask whether different regimes should lead to different portfolio choices.
- In practical portfolio management, regime thinking is useful because risk is not only about average return. It is also about drawdowns, volatility, correlation breakdowns, and whether the portfolio is exposed to the wrong risks at the wrong time.

This project applies that broad idea to an ETF allocation setting: use market and macroeconomic signals to estimate hidden market conditions, then test whether those estimated conditions can guide risk-aware allocation decisions.

### Project question

The central question is:

> Can market and macroeconomic signals be used to identify probabilistic market regimes, and can those regimes support a systematic ETF allocation strategy that improves risk-adjusted performance and drawdown control compared to simple static benchmarks?

In simpler words:

> Can we build a model that recognizes different market conditions, then use those conditions to decide when the portfolio should be more aggressive, more defensive, or more diversified?

### What this project is not trying to do

This project is not trying to prove that a machine learning model can perfectly predict the market.

It is also not trying to build a black-box trading system that automatically maximizes returns.

Instead, this project treats machine learning as a risk analysis tool. The Hidden Markov Model estimates the likely market regime. The strategy then uses transparent allocation rules for each regime. This distinction is important: the model helps describe the environment, while the allocation rule converts that environment into a portfolio decision.

### What success looks like

A useful result does not have to mean beating the stock market on raw return.

Because this is a risk-aware allocation project, success will be judged using metrics such as:

- lower maximum drawdown,
- lower volatility,
- better Sharpe or Sortino ratio,
- better behavior during stress periods,
- reasonable turnover after transaction costs,
- clear and economically interpretable regimes.

A strong project should also be honest if the model does not add value beyond simple benchmarks. In that case, the learning is still useful: it tells us that a more complex regime model must justify itself against simpler allocation rules.

## 2. Asset Universe and Data Sources

The asset universe matters because the model does not trade in isolation. Even if the regime model identifies a useful market condition, the strategy can only respond through the assets available in the portfolio.

For this project, the initial allocation universe is built around four liquid ETFs: SPY, TLT, GLD, and SHY. These ETFs are chosen because they represent different portfolio roles rather than four versions of the same risk exposure.

### Why these assets?

| ETF | Asset class | Portfolio role |
|---|---|---|
| SPY | U.S. equities | Growth and risk-on exposure |
| TLT | Long-term U.S. Treasuries | Duration exposure and potential recession hedge |
| GLD | Gold | Inflation, uncertainty, and alternative defensive exposure |
| SHY | Short-term U.S. Treasuries | Defensive cash-like exposure |

SPY represents broad U.S. equity market exposure. It is the main growth asset in the portfolio and is usually expected to do well when investors are willing to take risk.

TLT represents long-duration U.S. Treasury bonds. These bonds can perform well when growth slows and interest rates fall, but they can suffer when inflation and rising rates are the main concern.

GLD represents gold exposure. Gold does not behave like a stock or bond, so it can be useful when the market is worried about inflation, currency risk, or broader uncertainty.

SHY represents short-term U.S. Treasury exposure. It is included as a lower-volatility defensive asset, similar to a place where the strategy can reduce risk when the environment is unfavorable or unclear.

### Link to regime-based allocation

This asset universe connects directly to the regime allocation idea discussed earlier.

If regimes affect returns, volatility, and correlations, then the same portfolio may not be suitable in every environment. A calm growth regime may justify more equity exposure. A recession-like stress regime may favor Treasuries or short-duration defensive assets. An inflation or rate-pressure regime may require more caution toward long-duration bonds and may make gold or short-term Treasuries more useful.

This is why the project uses assets with different economic roles. The point is not just to identify regimes, but to test whether those regime estimates can lead to more sensible allocation choices.

### Data sources

This project uses two main data sources:

| Source | What it provides | How it is used |
|---|---|---|
| Tiingo | Historical ETF price data | Asset returns and market-based features |
| FRED | Macroeconomic and risk indicators | Macro context and regime features |

Tiingo provides the historical ETF prices needed to calculate portfolio returns. From these prices, we can engineer market features such as returns, momentum, rolling volatility, drawdowns, and correlations.

FRED provides macroeconomic and risk-related time series, such as interest rates, yield spreads, inflation measures, credit spreads, and recession indicators. These features help the model distinguish between market environments that may look similar from prices alone but are driven by different economic forces.

### Why these data sources are sufficient

The strategy is based on monthly allocation decisions, not high-frequency trading. Because of that, the project does not require intraday prices, order book data, or extremely detailed execution data.

Tiingo is sufficient for the market data side because the key raw input is adjusted historical ETF prices. Indicators such as volatility, momentum, trend, drawdown, and correlation do not need to come directly from the data provider. They can be calculated from the price history, which makes the feature definitions transparent and easier to check for lookahead bias.

FRED is sufficient for the macro side because it provides widely used public economic data. The main challenge is not finding the most complicated macro dataset, but using macro data carefully. In particular, macro features should be lagged so that the backtest does not accidentally use information that would not have been available at the time.

In short: Tiingo gives the market prices, FRED gives the economic context, and the project turns both into interpretable regime features.

## 3. Feature Engineering

Feature engineering is the step where raw data becomes useful model input.

The Hidden Markov Model does not directly understand labels such as "calm growth", "crisis", or "inflation stress". It only sees numerical columns. Those columns need to describe the market environment in a way that is both measurable and financially meaningful.

The goal of this section is to build a feature dataset that helps the model distinguish between different types of market conditions.

### Simple intuition

A useful analogy is a medical check-up.

A doctor does not judge someone's health from only one number. They may look at blood pressure, heart rate, cholesterol, symptoms, and medical history. Each measurement captures a different part of the picture.

Market regimes work similarly. A single signal, such as whether SPY is above its moving average, can be useful, but it only describes one part of the market. A richer regime model should look at several dimensions: returns, volatility, drawdowns, correlations, interest rates, credit stress, inflation, and broader macro conditions.

### Candidate feature groups

The candidate features are organized by what they are trying to measure.

| Feature group | Example features | What it tells us |
|---|---|---|
| Equity return and trend | SPY monthly return, 6-month momentum, 12-month momentum | Whether the main risky asset is rising or falling |
| Equity risk | SPY realized volatility, downside volatility, drawdown | Whether equity risk is calm or stressed |
| Cross-asset behavior | TLT return, GLD return, SPY-TLT correlation, SPY-GLD correlation | Whether diversification is working as expected |
| Volatility stress | VIX level, VIX change | Whether markets expect higher uncertainty |
| Yield curve and rates | 10Y-2Y spread, 10Y-3M spread, Fed funds rate | Growth expectations and monetary policy pressure |
| Credit stress | Baa-Treasury spread or high-yield spread | Whether investors are demanding more compensation for credit risk |
| Inflation pressure | CPI inflation, inflation trend, breakeven inflation | Whether inflation is becoming a dominant market risk |
| Macro growth | industrial production, unemployment rate | Whether the broader economy is expanding or weakening |

Not every candidate feature has to be used in the final HMM. Some features may be more useful for interpreting regimes after the model is trained. The important idea is to start with a controlled set of economically meaningful signals, then decide which ones belong in the actual model.

### Raw data needed

Most market-derived features can be calculated from ETF price history. For example, returns, momentum, volatility, drawdowns, and correlations can all be derived from adjusted prices.

The macro and risk features come from FRED. These series help identify whether the market environment is being shaped by growth risk, inflation pressure, credit stress, or monetary policy.

| Data source | Raw data to retrieve | Features created from it |
|---|---|---|
| Tiingo | daily adjusted prices for SPY, TLT, GLD, SHY | returns, momentum, realized volatility, drawdown, correlations |
| FRED | VIX, yield spreads, policy rates, inflation, credit spreads, growth indicators | volatility stress, rate pressure, inflation trend, credit stress, macro growth |

The final dataset should be monthly, because the strategy will make monthly allocation decisions.

### Implementation

The following code collects raw data from Tiingo and FRED, converts it into monthly market and macro features, and combines everything into a single feature matrix.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Display settings make notebook tables easier to read.
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# If the notebook is opened from the notebooks folder, move one level up.
# If it is opened from the project root, keep the current folder.
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# Load API keys from the local .env file.
# The keys are used by the code, but never printed in the notebook.
load_dotenv(project_root / ".env")
TIINGO_API_KEY = os.getenv("TIINGO_API_KEY")
FRED_API_KEY = os.getenv("FRED_API_KEY")

# Local cache folders prevent repeated API calls every time the notebook is rerun.
RAW_DATA_DIR = project_root / "data" / "raw"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Set this to True only when you intentionally want to refresh the API data.
REFRESH_DATA = False

ETF_PRICE_CACHE = RAW_DATA_DIR / "tiingo_etf_prices.csv"
FRED_RAW_CACHE = RAW_DATA_DIR / "fred_macro_series.csv"
FEATURE_MATRIX_CACHE = PROCESSED_DATA_DIR / "feature_matrix.csv"
ASSET_RETURNS_CACHE = PROCESSED_DATA_DIR / "asset_returns.csv"

# Start in 2005 because GLD started trading in late 2004,
# so 2005 gives us a cleaner shared ETF history.
START_DATE = "2005-01-01"
# Use an explicit analysis end date so results are reproducible.
END_DATE = "2026-05-31"

ETF_TICKERS = ["SPY", "TLT", "GLD", "SHY"]

# FRED series used as candidate macro/risk inputs.
# We may not use every series in the final HMM, but they are useful candidates.
FRED_SERIES = {
    "VIXCLS": "vix",
    "T10Y2Y": "term_spread_10y2y",
    "T10Y3M": "term_spread_10y3m",
    "FEDFUNDS": "fed_funds_rate",
    "CPIAUCSL": "cpi",
    "T10YIE": "breakeven_inflation_10y",
    "BAA10Y": "credit_spread_baa10y",
    "INDPRO": "industrial_production",
    "UNRATE": "unemployment_rate",
}

print(f"Project root: {project_root}")
print(f"Tiingo key available: {TIINGO_API_KEY is not None}")
print(f"FRED key available: {FRED_API_KEY is not None}")
print(f"Refresh API data: {REFRESH_DATA}")

This setup cell defines the project paths, date range, ETF tickers, and FRED series. It also creates local cache folders. The `REFRESH_DATA` flag controls whether the notebook downloads fresh data from the APIs or reuses locally saved CSV files.

In [ ]:
def fetch_tiingo_prices(ticker, start_date=START_DATE, end_date=END_DATE):
    """Download daily adjusted close prices for one ETF from Tiingo."""
    if TIINGO_API_KEY is None:
        raise ValueError("Missing TIINGO_API_KEY. Add it to your local .env file.")

    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    headers = {"Authorization": f"Token {TIINGO_API_KEY}"}
    params = {"startDate": start_date}

    if end_date is not None:
        params["endDate"] = end_date

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    if not data:
        raise ValueError(f"No Tiingo data returned for {ticker}.")

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.set_index("date").sort_index()

    # adjClose adjusts for dividends and splits, which is what we want for returns.
    return df["adjClose"].rename(ticker)


def fetch_fred_series(series_id, start_date=START_DATE, end_date=END_DATE):
    """Download one time series from FRED."""
    if FRED_API_KEY is None:
        raise ValueError("Missing FRED_API_KEY. Add it to your local .env file.")

    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start_date,
    }

    if end_date is not None:
        params["observation_end"] = end_date

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    observations = response.json()["observations"]
    df = pd.DataFrame(observations)
    df["date"] = pd.to_datetime(df["date"])

    # FRED sometimes uses '.' for missing observations, so convert safely to NaN.
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.set_index("date").sort_index()

    return df["value"].rename(series_id)

These helper functions keep the API logic in one place. The Tiingo function returns one adjusted-price series for an ETF. The FRED function returns one cleaned macro series, converting missing values into `NaN` and making the date column the index.

In [ ]:
# Load cached ETF prices if available; otherwise download from Tiingo.
if ETF_PRICE_CACHE.exists() and not REFRESH_DATA:
    etf_prices = pd.read_csv(ETF_PRICE_CACHE, index_col="date", parse_dates=True)
    print("Loaded ETF prices from local cache.")
else:
    etf_prices = pd.concat(
        [fetch_tiingo_prices(ticker) for ticker in ETF_TICKERS],
        axis=1,
    ).sort_index()
    etf_prices.to_csv(ETF_PRICE_CACHE, index_label="date")
    print("Downloaded ETF prices from Tiingo and saved them locally.")

# Load cached macro data if available; otherwise download from FRED.
if FRED_RAW_CACHE.exists() and not REFRESH_DATA:
    fred_raw = pd.read_csv(FRED_RAW_CACHE, index_col="date", parse_dates=True)
    print("Loaded FRED series from local cache.")
else:
    fred_raw = pd.concat(
        [fetch_fred_series(series_id) for series_id in FRED_SERIES],
        axis=1,
    ).rename(columns=FRED_SERIES).sort_index()
    fred_raw.to_csv(FRED_RAW_CACHE, index_label="date")
    print("Downloaded FRED series and saved them locally.")

print(f"ETF price shape: {etf_prices.shape}")
print(f"FRED raw shape: {fred_raw.shape}")

display(etf_prices.tail())
display(fred_raw.tail())

This cell uses a cache-first workflow. If the local CSV files already exist, the notebook loads them directly and avoids calling the APIs again. If the files do not exist, or if `REFRESH_DATA` is set to `True`, the notebook downloads fresh data and saves a local copy.

In [ ]:
# Convert daily adjusted prices to monthly prices using the last available price each month.
monthly_prices = etf_prices.resample("ME").last()

# Monthly returns are used both as candidate features and later for backtesting.
monthly_returns = monthly_prices.pct_change()
daily_returns = etf_prices.pct_change()

market_features = pd.DataFrame(index=monthly_prices.index)

# Equity return and trend features.
market_features["spy_return_1m"] = monthly_returns["SPY"]
market_features["spy_momentum_6m"] = monthly_prices["SPY"].pct_change(6)
market_features["spy_momentum_12m"] = monthly_prices["SPY"].pct_change(12)

# Drawdown measures how far SPY is below its previous high.
market_features["spy_drawdown"] = monthly_prices["SPY"] / monthly_prices["SPY"].cummax() - 1

# Realized volatility is calculated from daily returns, then sampled at month-end.
# 63 trading days is roughly 3 months; annualization uses sqrt(252).
market_features["spy_realized_vol_3m"] = (
    daily_returns["SPY"].rolling(63).std() * np.sqrt(252)
).resample("ME").last()

# Cross-asset returns help describe whether bonds/gold are acting differently from equities.
market_features["tlt_return_1m"] = monthly_returns["TLT"]
market_features["gld_return_1m"] = monthly_returns["GLD"]
market_features["shy_return_1m"] = monthly_returns["SHY"]

# Rolling correlations help measure whether diversification is working as expected.
# 126 trading days is roughly 6 months.
market_features["spy_tlt_corr_6m"] = (
    daily_returns["SPY"].rolling(126).corr(daily_returns["TLT"])
).resample("ME").last()

market_features["spy_gld_corr_6m"] = (
    daily_returns["SPY"].rolling(126).corr(daily_returns["GLD"])
).resample("ME").last()

display(market_features.tail())

The market features are calculated from ETF prices. Monthly return measures the most recent asset performance. Momentum measures performance over longer windows. Drawdown measures how far SPY is below its previous high. Realized volatility measures how much SPY has been moving day to day. Rolling correlations measure whether Treasuries or gold are moving with or against equities.

In [ ]:
# Convert FRED series to monthly frequency using the last available observation each month.
# Forward-fill handles series that are not reported every trading day.
fred_monthly = fred_raw.resample("ME").last().ffill()

# Lag macro features by one month to reduce lookahead bias.
# This is a conservative simplification because macro data is often released after month-end.
macro_input = fred_monthly.shift(1)

macro_features = pd.DataFrame(index=macro_input.index)

# Volatility stress.
macro_features["vix_level"] = macro_input["vix"]
macro_features["vix_change_1m"] = macro_input["vix"].diff(1)

# Yield curve and policy rate conditions.
macro_features["term_spread_10y2y"] = macro_input["term_spread_10y2y"]
macro_features["term_spread_10y3m"] = macro_input["term_spread_10y3m"]
macro_features["fed_funds_rate"] = macro_input["fed_funds_rate"]
macro_features["fed_funds_change_3m"] = macro_input["fed_funds_rate"].diff(3)

# Credit stress.
macro_features["credit_spread_baa10y"] = macro_input["credit_spread_baa10y"]
macro_features["credit_spread_change_3m"] = macro_input["credit_spread_baa10y"].diff(3)

# Inflation pressure.
macro_features["inflation_yoy"] = macro_input["cpi"].pct_change(12) * 100
macro_features["inflation_trend_3m"] = macro_features["inflation_yoy"].diff(3)
macro_features["breakeven_inflation_10y"] = macro_input["breakeven_inflation_10y"]

# Macro growth context.
macro_features["industrial_production_yoy"] = macro_input["industrial_production"].pct_change(12) * 100
macro_features["unemployment_rate"] = macro_input["unemployment_rate"]
macro_features["unemployment_change_3m"] = macro_input["unemployment_rate"].diff(3)

display(macro_features.tail())

The macro features are first converted to month-end observations. They are then shifted by one month so the model does not accidentally use macro information that may not have been available at the time. The resulting features describe volatility stress, yield curve conditions, monetary policy pressure, credit stress, inflation pressure, and macro growth.

In [ ]:
# Combine market and macro features into one monthly dataset.
candidate_features = pd.concat([market_features, macro_features], axis=1).sort_index()

# Drop rows with missing values after rolling windows, percentage changes, and macro lags.
# The HMM needs a complete feature matrix with no NaNs.
feature_matrix = candidate_features.dropna().copy()

# Keep monthly returns separately for later backtesting.
# These are not all necessarily model features; they will also be used to calculate portfolio performance.
asset_returns = monthly_returns.loc[feature_matrix.index].copy()

# Save processed datasets locally so later notebook sections can reuse them.
feature_matrix.to_csv(FEATURE_MATRIX_CACHE, index_label="date")
asset_returns.to_csv(ASSET_RETURNS_CACHE, index_label="date")

print(f"Candidate feature matrix shape: {feature_matrix.shape}")
print(f"Feature sample: {feature_matrix.index.min().date()} to {feature_matrix.index.max().date()}")

display(feature_matrix.tail())

The final output of this section is `feature_matrix`, where each row is one month and each column is a candidate regime feature. Rows with missing values are removed because the HMM will require complete inputs. `asset_returns` is kept separately because it will be used later to calculate strategy and benchmark performance.

## 4. Hidden Markov Model Explanation

The previous section created the `feature_matrix`: a monthly table of market and macro signals. The next question is how to turn those signals into an estimate of the current market regime.

A Hidden Markov Model, or HMM, is useful here because it is designed for situations where we observe signals over time, but the underlying state that generated those signals is not directly visible.

### Observed signals vs hidden states

In this project, the observed signals are the features we calculated earlier:

- returns,
- momentum,
- volatility,
- drawdowns,
- correlations,
- VIX,
- yield spreads,
- credit spreads,
- inflation,
- policy rates.

The hidden states are the market regimes we are trying to infer.

There is no official label that says a particular month is definitely a "calm growth regime" or an "inflation stress regime". Those regime labels are not directly observed. They are estimated from the data.

### From Markov chains to Hidden Markov Models

This idea is closely related to Markov chains, which are often introduced in linear algebra and probability courses.

A Markov chain models a system that moves between states over time. For example, a market could move between states such as:

```text
calm -> stress -> recovery
```

The key idea is the Markov property:

> The next state depends on the current state, not the full history before it.

For example, if the market is currently calm, a Markov chain might estimate:

```text
85% chance the next month remains calm
10% chance the next month becomes stressed
5% chance the next month enters another state
```

A Hidden Markov Model builds on this idea. It still models transitions between states, but the states are hidden. Instead of directly observing the state, we observe data generated by that state.

### Simple intuition

A useful analogy is a weather station.

Suppose we cannot directly observe the true weather condition, but we can observe sensor readings:

- temperature,
- humidity,
- wind speed,
- air pressure,
- rainfall.

From these readings, we might infer whether the hidden weather condition is calm, stormy, humid, or cold.

Markets are similar. We cannot directly observe the true market regime, but we can observe market and macro signals. The HMM uses those signals to estimate which hidden market state most likely generated them.

### The two parts of an HMM

An HMM has two main pieces:

| Component | Meaning in simple terms | Market interpretation |
|---|---|---|
| Transition model | How hidden states move from one period to the next | How likely regimes are to persist or switch |
| Emission model | What observations each hidden state tends to produce | What returns, volatility, spreads, and macro signals look like in each regime |

The transition model captures the idea that regimes tend to persist. A calm market usually does not randomly become a crisis regime for one month and then immediately switch back without any pattern.

The emission model captures the idea that each regime tends to produce different observable data. For example, a stress regime may produce higher volatility, weaker equity returns, and wider credit spreads.

### HMM vs k-means clustering

A useful comparison is k-means clustering.

K-means groups observations based on similarity. If two months have similar feature values, k-means may place them in the same cluster. However, k-means treats each month independently. It does not know whether one month came before or after another.

An HMM also groups observations into hidden states, but it adds time dependence. It considers both:

- what the current data looks like, and
- what regime the market was likely in previously.

A simple way to say this is:

> K-means treats the market like a set of photographs. An HMM treats the market like a movie.

That time dimension matters because market regimes usually have persistence. If the market has been calm for several months, one slightly volatile month may raise the probability of stress, but it does not necessarily mean the entire regime has changed immediately.

### What the HMM learns

In this project, the HMM will learn three main things:

1. **State characteristics**: what each hidden regime tends to look like in terms of returns, volatility, correlations, spreads, inflation, and other features.
2. **Transition probabilities**: how likely the market is to stay in the same regime or move to another regime.
3. **Regime probabilities**: for each month, the probability that the market belongs to each hidden state.

This last point is important. The HMM does not produce objective market labels. It produces probabilistic estimates based on the data and model assumptions.

For example, the model may estimate that a particular month has:

```text
10% probability of Regime 0
20% probability of Regime 1
70% probability of Regime 2
```

We may later interpret Regime 2 as an inflation or rate-pressure regime, but that interpretation comes after studying the data associated with the hidden state.

### How this connects to the project

The HMM sits between feature engineering and portfolio allocation.

The flow is:

```text
feature_matrix
    -> HMM
        -> regime probabilities
            -> regime interpretation
                -> allocation rule
                    -> backtest
```

The model does not directly decide the portfolio weights. Instead, it estimates the likely market regime. After that, we interpret the regimes and use transparent allocation rules to decide how much to hold in SPY, TLT, GLD, and SHY.

This keeps the project aligned with the risk analytics goal: the HMM is used to understand the market environment, not as a black-box return prediction machine.

## 5. Regime Model Training

The previous section explained what an HMM is conceptually. This section turns that idea into a trained model.

The goal is not to assume the market has a fixed number of regimes upfront. Instead, we train three candidate HMMs with 2, 3, and 4 hidden states, then compare whether the extra complexity seems justified.

At this stage, the regimes are still just numbered hidden states. We will interpret what those states mean economically in the next section.

### Load the modelling data

We start from the two processed files created in the feature engineering section. The `feature_matrix` contains the market and macro signals used by the HMM. The `asset_returns` table is loaded as well, but it is not used to train the HMM. It is kept for later, when we test whether regime-aware allocation rules improve portfolio behaviour.

In [ ]:
from pathlib import Path
import logging

# hmmlearn can log tiny numerical convergence messages for random starts that are not kept.
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# Resolve the project root whether the notebook is opened from the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_MATRIX_CACHE = PROCESSED_DATA_DIR / "feature_matrix.csv"
ASSET_RETURNS_CACHE = PROCESSED_DATA_DIR / "asset_returns.csv"
MODEL_COMPARISON_CACHE = PROCESSED_DATA_DIR / "regime_model_comparison.csv"

if not FEATURE_MATRIX_CACHE.exists() or not ASSET_RETURNS_CACHE.exists():
    raise FileNotFoundError(
        "Processed data was not found. Run the feature engineering section first "
        "to create feature_matrix.csv and asset_returns.csv."
    )

feature_matrix = pd.read_csv(FEATURE_MATRIX_CACHE, parse_dates=["date"], index_col="date")
asset_returns = pd.read_csv(ASSET_RETURNS_CACHE, parse_dates=["date"], index_col="date")

# Sorting protects us from accidental date-order issues after loading from CSV.
feature_matrix = feature_matrix.sort_index()
asset_returns = asset_returns.sort_index()

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Asset returns shape: {asset_returns.shape}")
print(f"Sample period: {feature_matrix.index.min().date()} to {feature_matrix.index.max().date()}")

display(feature_matrix.head())

This code reloads the clean monthly datasets from the local cache. The HMM only receives the feature matrix, because it is trying to infer market conditions from signals such as returns, volatility, yield spreads, inflation, credit stress, and unemployment. Asset returns are deliberately kept separate so that later performance testing does not accidentally become part of model training.

### Split the sample by time

For normal machine learning examples, it is common to randomly split rows into training and test sets. That would be inappropriate here because financial data has a time order. A random split would allow the model setup to learn from future periods while pretending to evaluate the past.

Instead, we split the data chronologically:

```text
Training period: used to fit the HMM parameters
Validation period: used to compare the 2-state, 3-state, and 4-state candidates
Test period: left aside for the later backtest section
```

The training period runs from 2006 to 2018, so it includes the global financial crisis, the post-crisis recovery, the eurozone debt stress period, the low-rate quantitative easing environment, and the volatility/rate-hike stress of 2018. The validation period covers 2019 to 2021, including the COVID crash, the rapid policy response, the reopening period, and the early phase of inflation pressure. The test period begins in 2022, which includes the stock-bond drawdown, aggressive rate hikes, persistent inflation stress, and later geopolitical and macro uncertainty.

This is still not the final trading simulation. The fully realistic version will be the walk-forward backtest later, where the model is repeatedly refit using only information available at that point in time.

In [ ]:
TRAIN_END = pd.Timestamp("2018-12-31")
VALIDATION_END = pd.Timestamp("2021-12-31")

train_features = feature_matrix.loc[feature_matrix.index <= TRAIN_END].copy()
validation_features = feature_matrix.loc[
    (feature_matrix.index > TRAIN_END) & (feature_matrix.index <= VALIDATION_END)
].copy()
test_features = feature_matrix.loc[feature_matrix.index > VALIDATION_END].copy()

split_summary = pd.DataFrame(
    {
        "start": [train_features.index.min(), validation_features.index.min(), test_features.index.min()],
        "end": [train_features.index.max(), validation_features.index.max(), test_features.index.max()],
        "months": [len(train_features), len(validation_features), len(test_features)],
    },
    index=["train", "validation", "test"],
)

print(split_summary)

The split keeps the timeline intact. The training period is where the model learns the statistical behaviour of the hidden states. The validation period helps us compare candidate regime counts without touching the later test period. The test period is left untouched because it will be more meaningful in the walk-forward backtest.

### Standardise the features without using future information

The feature columns are measured in different units. Monthly returns, VIX levels, inflation changes, yield spreads, and unemployment rates do not naturally live on the same scale. If we feed them into the HMM directly, large-number variables can dominate smaller-number variables.

To avoid this, we standardise each feature by subtracting its mean and dividing by its standard deviation. The important detail is that the scaler is fitted on the training period only, then reused on validation and test data.

In [ ]:
scaler = StandardScaler()

# Fit the scaler only on the training period to avoid using future information.
X_train = scaler.fit_transform(train_features)
X_validation = scaler.transform(validation_features)
X_test = scaler.transform(test_features)
X_all = scaler.transform(feature_matrix)

scaled_train_features = pd.DataFrame(X_train, index=train_features.index, columns=train_features.columns)
scaled_validation_features = pd.DataFrame(X_validation, index=validation_features.index, columns=validation_features.columns)
scaled_test_features = pd.DataFrame(X_test, index=test_features.index, columns=test_features.columns)
scaled_all_features = pd.DataFrame(X_all, index=feature_matrix.index, columns=feature_matrix.columns)

# After standardisation, training features should have mean near 0 and standard deviation near 1.
scaling_check = pd.DataFrame(
    {
        "train_mean": scaled_train_features.mean().round(3),
        "train_std": scaled_train_features.std(ddof=0).round(3),
    }
)

display(scaling_check.head(10))

This cell puts all model inputs onto a comparable scale. The scaler is fitted only on the training period and then reused on the validation and test periods, which avoids letting future market conditions influence how past data is transformed.

### Define model comparison helpers

We will train Gaussian HMMs. This means each hidden state is assumed to generate feature values from a normal distribution. We also use diagonal covariance matrices, which is a practical choice for a modest monthly dataset because it keeps the number of parameters manageable.

Each candidate model is trained several times with different random seeds. This matters because HMM training can start from different initial guesses and occasionally settle into weaker solutions. To keep the setup transparent, we initialise each model with simple starting values instead of relying on hidden library defaults.

In [ ]:
STATE_COUNTS = [2, 3, 4]
RANDOM_SEEDS = range(20)
N_ITER = 1000
TOLERANCE = 1e-3


def count_hmm_parameters(n_states, n_features):
    """Count free parameters for a Gaussian HMM with diagonal covariance."""
    start_prob_params = n_states - 1
    transition_params = n_states * (n_states - 1)
    mean_params = n_states * n_features
    variance_params = n_states * n_features
    return start_prob_params + transition_params + mean_params + variance_params


def calculate_information_criteria(log_likelihood, n_states, n_features, n_observations):
    """Calculate AIC and BIC so models with different state counts can be compared."""
    n_params = count_hmm_parameters(n_states, n_features)
    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_observations)
    return aic, bic, n_params


def fit_hmm_candidate(n_states, random_seed, X):
    """Fit one HMM candidate for a given state count and random seed."""
    rng = np.random.default_rng(random_seed)

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=N_ITER,
        tol=TOLERANCE,
        min_covar=1e-3,
        random_state=random_seed,
        init_params="",  # We set initial values ourselves so the setup is explicit.
    )

    # Start with equal regime probabilities.
    model.startprob_ = np.repeat(1 / n_states, n_states)

    # Start with persistent regimes: high probability of staying in the same state.
    off_diagonal_probability = 0.05 / (n_states - 1)
    model.transmat_ = np.full((n_states, n_states), off_diagonal_probability)
    np.fill_diagonal(model.transmat_, 0.95)

    # Use random training months as initial state means.
    initial_mean_rows = rng.choice(len(X), size=n_states, replace=False)
    model.means_ = X[initial_mean_rows].copy()

    # Start every state with the overall feature variance.
    model.covars_ = np.tile(np.var(X, axis=0) + 1e-3, (n_states, 1))

    model.fit(X)
    return model

The helper functions keep the model training code readable. `count_hmm_parameters` and `calculate_information_criteria` are used because a model with more states will usually fit the training data better, but it also has more parameters. AIC and BIC add a penalty for extra complexity, which helps us avoid choosing a more complicated model just because it memorises the sample better.

### Train 2-state, 3-state, and 4-state HMMs

Now we fit the candidate models. For each number of hidden states, we try multiple random seeds and keep the strongest run. We record both statistical fit and practical diagnostics, such as whether one state is used for only a tiny fraction of the sample.

A regime model is not useful just because it has a high likelihood. It also needs to produce states that are stable enough and common enough to interpret.

In [ ]:
model_results = []
best_models = {}

n_features = X_train.shape[1]
n_train_observations = X_train.shape[0]

for n_states in STATE_COUNTS:
    best_model = None
    best_train_log_likelihood = -np.inf

    for seed in RANDOM_SEEDS:
        model = fit_hmm_candidate(n_states=n_states, random_seed=seed, X=X_train)
        train_log_likelihood = model.score(X_train)

        # Keep the best random initialisation for this state count.
        if train_log_likelihood > best_train_log_likelihood:
            best_train_log_likelihood = train_log_likelihood
            best_model = model

    train_log_likelihood = best_model.score(X_train)
    validation_log_likelihood = best_model.score(X_validation)
    aic, bic, n_params = calculate_information_criteria(
        log_likelihood=train_log_likelihood,
        n_states=n_states,
        n_features=n_features,
        n_observations=n_train_observations,
    )

    train_states = best_model.predict(X_train)
    state_counts = pd.Series(train_states).value_counts(normalize=True).sort_index()
    min_state_share = state_counts.min()
    max_state_share = state_counts.max()

    # The diagonal of the transition matrix shows how likely each state is to persist.
    average_self_transition = np.diag(best_model.transmat_).mean()

    model_results.append(
        {
            "n_states": n_states,
            "best_train_log_likelihood": train_log_likelihood,
            "train_log_likelihood_per_month": train_log_likelihood / len(X_train),
            "validation_log_likelihood": validation_log_likelihood,
            "validation_log_likelihood_per_month": validation_log_likelihood / len(X_validation),
            "aic": aic,
            "bic": bic,
            "n_parameters": n_params,
            "converged": best_model.monitor_.converged,
            "iterations": best_model.monitor_.iter,
            "min_state_share_train": min_state_share,
            "max_state_share_train": max_state_share,
            "average_self_transition": average_self_transition,
        }
    )
    best_models[n_states] = best_model

model_comparison = pd.DataFrame(model_results).sort_values("n_states")
model_comparison.to_csv(MODEL_COMPARISON_CACHE, index=False)

display(model_comparison.round(4))

This table is the first serious comparison between candidate regime counts.

A few rules of thumb help interpret the columns:

- Higher training log likelihood means the model fits the training period better.
- Higher validation log likelihood is more important because it suggests the model explains later unseen data better.
- Lower AIC and BIC are better because they reward fit but penalise extra parameters.
- `min_state_share_train` should not be too close to zero; otherwise, the model may have created a regime that is barely used.
- `average_self_transition` measures persistence. Higher values mean regimes tend to last longer instead of switching every month.

In this run, the 4-state model fits the training period best, which is expected because it has more flexibility. However, the 3-state model has the strongest validation log likelihood, while the 4-state validation result is weaker. That is an early warning that the 4-state version may be overfitting. We still do not choose the final model from this table alone; the next section will check whether the states make economic sense.

### Generate diagnostic regime probabilities

The most useful output of an HMM is not only the single most likely state. It is the probability assigned to each hidden state each month.

For example, a month might be 80% likely to belong to a stressed regime and 20% likely to belong to a normal regime. That is more informative than pretending the market condition is known with certainty.

The probabilities below are diagnostic model outputs for interpretation. The final trading backtest will still need a walk-forward design so that allocation decisions only use information available at the time.

In [ ]:
regime_probability_outputs = {}
probability_diagnostics = []

for n_states, model in best_models.items():
    probabilities = model.predict_proba(X_all)
    probability_columns = [f"state_{state}_probability" for state in range(n_states)]

    regime_probabilities = pd.DataFrame(
        probabilities,
        index=feature_matrix.index,
        columns=probability_columns,
    )

    # The most likely state is a convenient summary, but the full probability distribution is more informative.
    regime_probabilities["most_likely_state"] = np.argmax(probabilities, axis=1)
    regime_probability_outputs[n_states] = regime_probabilities

    output_path = PROCESSED_DATA_DIR / f"regime_probabilities_{n_states}_states.csv"
    regime_probabilities.to_csv(output_path, index_label="date")

    max_probability = regime_probabilities[probability_columns].max(axis=1)
    state_shares = regime_probabilities["most_likely_state"].value_counts(normalize=True).sort_index()

    for state, share in state_shares.items():
        probability_diagnostics.append(
            {
                "n_states": n_states,
                "diagnostic": f"state_{state}_share",
                "value": share,
            }
        )

    probability_diagnostics.extend(
        [
            {
                "n_states": n_states,
                "diagnostic": "average_highest_probability",
                "value": max_probability.mean(),
            },
            {
                "n_states": n_states,
                "diagnostic": "lowest_highest_probability",
                "value": max_probability.min(),
            },
        ]
    )

probability_diagnostics = pd.DataFrame(probability_diagnostics)

# Use the 3-state candidate as an example because it is often the most interpretable middle ground.
example_probabilities = regime_probability_outputs[3].copy()
example_probability_columns = [
    column for column in example_probabilities.columns if column.endswith("_probability")
]
example_probabilities["highest_probability"] = example_probabilities[example_probability_columns].max(axis=1)
least_certain_examples = example_probabilities.sort_values("highest_probability").head(10)

print("Saved diagnostic regime probability files:")
for n_states in STATE_COUNTS:
    print(f"- regime_probabilities_{n_states}_states.csv")

display(probability_diagnostics.pivot(index="diagnostic", columns="n_states", values="value").round(4))
display(least_certain_examples.round(4))

This cell saves the monthly regime probabilities for the 2-state, 3-state, and 4-state candidates. The state numbers are just model labels for now. `state_0` does not automatically mean good or bad; we need to examine the feature behaviour and asset returns associated with each state before attaching economic meaning.

The second table shows months where the 3-state model is least certain. These examples are useful because they make it clear that the HMM produces probabilities, not just hard regime labels.

That interpretation step comes next.